In [ ]:
import numpy as np
from datetime import date
from matplotlib import pyplot as plt
import marineHeatWaves as mhw
import xarray as xr

In [ ]:
ds = xr.open_dataset('path/OISST_daily_mean_1982_2022.nc')
sst_ = ds.sst.sel(time=slice('1998', '2022'), lat=slice(-10,30), lon=slice(40,120))
t = np.arange(date(1998,1,1).toordinal(),date(2022,12,31).toordinal()+1)
sst = sst_.values

sst.shape

In [ ]:
nan_mask = np.isnan(sst)
sst_without_nan = sst.copy()
sst_without_nan[nan_mask] = 0

nan_mask.shape, sst_without_nan.shape

In [ ]:
mhws_dict = {}
clim_dict = {}

# Loop through latitudes and longitudes
for i in range(len(sst_.lat)):
    for j in range(len(sst_.lon)):
        mhws, clim = mhw.detect(t, sst_without_nan[:, i, j])

        # Store dictionaries in the dictionary of lists
        mhws_dict[(i, j)] = mhws_dict.get((i, j), []) + [mhws]
        clim_dict[(i, j)] = clim_dict.get((i, j), []) + [clim]



In [ ]:
np.save('/files_npy/OBS_mhw_dict_1998_2022.npy', mhws_dict)
np.save('/files_npy/OBS_clim_dict_1998_2022.npy', clim_dict)

In [ ]:
mhws_dict = np.load('path/OBS_mhw_dict_1998_2022.npy', allow_pickle=True).item()

In [ ]:
mhwBlock_dict = {}

# Loop through latitudes and longitudes
for i in range(len(sst_.lat)):
    for j in range(len(sst_.lon)):
        mhwBlock = mhw.blockAverage(t, mhws_dict[(i,j)][0])

        # Store dictionaries in the dictionary of lists
        mhwBlock_dict[(i,j)] = mhwBlock.get((i,j), []) + [mhwBlock]

In [ ]:
np.save('/files_npy/OBS_mhwBlock_dict_1998_2022.npy', mhwBlock_dict)

In [ ]:
mhwBlock_dict = np.load('path/OBS_mhwBlock_dict_1998_2022.npy', allow_pickle=True).item()

In [ ]:
mean_dict = {}; trend_dict = {}; dtrend_dict = {}

# Loop through latitudes and longitudes
for i in range(len(sst_.lat)):
    for j in range(len(sst_.lon)):
        mean, trend, dtrend = mhw.meanTrend(mhwBlock_dict[(i,j)][0])

        # Store dictionaries in the dictionary of lists
        mean_dict[(i,j)] = mean_dict.get((i,j), []) + [mean]
        trend_dict[(i,j)] = trend_dict.get((i,j), []) + [trend]
        dtrend_dict[(i,j)] = dtrend_dict.get((i,j), []) + [dtrend]



In [ ]:
np.save('/files_npy/OBS_mean_dict_1998_2022.npy', mean_dict)
np.save('/files_npy/OBS_trend_dict_1998_2022.npy', trend_dict)
np.save('/files_npy/OBS_dtrend_dict_1998_2022.npy', dtrend_dict)